In [1]:
import os
import re
from itertools import chain
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import spacy
from spacy.tokens.doc import Doc
from spacy.tokens.span import Span
from spacy.tokens.token import Token
import coreferee
from coreferee.data_model import ChainHolder
from helpers import get_request
from bs4 import BeautifulSoup
from bs4.element import Tag

c:\Users\Adam\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\coreferee\manager.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("coreferee")

In [3]:
def get_text_from_file(path: str):
    text = ""
    
    try:
        with open(path, "r", encoding="utf8") as file:
            text = file.read()
    except Exception as e:
        print("Error reading file", path)
        print(e)
    finally:
        return text

In [4]:
articles_folder = "./articles"

In [5]:
article_files = os.listdir(articles_folder)
article_files = [file for file in article_files if file.endswith(".txt")]

In [6]:
files_subset = article_files[:5]

In [7]:
# TODO: run on entire dataset
# df = pd.DataFrame(data={
#     "file": article_files,
#     "text": [get_text_from_file(f"{articles_folder}/{file}") for file in article_files]
# })

texts = [get_text_from_file(f"{articles_folder}/{file}") for file in files_subset]

df = pd.DataFrame(data={
    "file": files_subset,
    "text": texts
})
df

,file,text
0,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ..."
1,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...
2,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu..."
3,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...
4,news_10-most-memorable-ufc-moments-canada.txt,As we continue our Canadian-focused march thro...


In [8]:
# This takes a long time, so have patience
df["doc"] = list(nlp.pipe(texts))

In [9]:
def get_people(doc: Doc):
    return [ent for ent in doc.ents if ent.label_ == "PERSON"]

In [10]:
df["people"] = df["doc"].apply(get_people)

In [11]:
all_people: list[Span] = list(chain.from_iterable(df["people"]))
all_names = set(p.text for p in all_people)
full_names = set(name for name in all_names 
                 if 
                 len(name.split()) >= 2 
                 and len(name.split()) <= 3
                 and re.search("^[a-zA-Z\s’]+$", name) is not None)

In [12]:
def is_woman_tag(tag: Tag):
    text = tag.get_text().strip().lower()
    return "woman" in text or "women" in text

def find_gender_on_ufc(name: str):
    name_query = "-".join(name.split()).replace('’', ' ')
    url = f"https://www.ufc.com/athlete/{name_query}"

    try:
        response = get_request(url)
        soup = BeautifulSoup(response.text, "html.parser")

        tags: list[Tag] = soup.find_all("p", "hero-profile__tag")
        if not tags:
            return None

        is_woman = any([is_woman_tag(tag) for tag in tags])
        if is_woman:
            return "woman"
        else:
            return "man"
    except Exception as e:
        print("Error getting gender of", name)
        print(e)

In [29]:
full_names

{'Al Iaquinta',
 'Alex Pereira',
 'Alexa Grasso',
 'Alexander Volkanovski',
 'Alexandre Pantoja',
 'Amanda Ribas',
 'Anderson Silva',
 'Andrea Lee',
 'Andreas Michailidis',
 'Anthony Perosh',
 'Antonio Rogerio Nogueira',
 'Arman Tsarukyan',
 'Belal Muhammad',
 'Bo Nickal',
 'Brad Tavares',
 'Brandon Moreno',
 'Brandon Royval',
 'Bruce Buffer',
 'Cain Velasquez',
 'Chael Sonnen',
 'Charles Oliveira',
 'Chris Clements',
 'Chris Leben',
 'Chris Weidman',
 'Christos Giagos',
 'Clint Hester',
 'Cody Garbrandt',
 'Colby Covington',
 'Cory Sandhagen',
 'Dan Hooker',
 'Dan Kelly',
 'Dana White',
 'Dana White’s',
 'Daniel Cormier',
 'Dave Menne',
 'Deiveson Figueiredo',
 'Della Maddalena',
 'Derek Brunson',
 'Dione Barbosa',
 'Dominick Cruz',
 'Dos Santos',
 'Drakkar Klose',
 'Dricus Du Plessis',
 'Dricus du Plessis',
 'Dustin Poirier',
 'Elias Theodorou',
 'Eric Nicksick',
 'Erin Blanchfield',
 'Forrest Griffin',
 'Gerald Meerschaert',
 'Gil Castillo',
 'Gilbert Burns',
 'Glover Teixeira',
 'I

In [ ]:
# Set this to False if you want to fetch the genders from the UFC website
use_saved_gender_map = False
genders_map_file = "./genders_map.csv"

if use_saved_gender_map and os.path.exists(genders_map_file):
    name_gender_pairs_df = pd.read_csv(genders_map_file)
    genders_map: dict[str, str] = dict(zip(name_gender_pairs_df["name"], name_gender_pairs_df["gender"]))
else:
    def get_name_gender_pair_on_ufc(name: str):
        return (name, find_gender_on_ufc(name))

    with ThreadPoolExecutor() as executor:
        name_gender_pairs = [
            (n, g)
            for n, g in executor.map(get_name_gender_pair_on_ufc, full_names)
            if g is not None
        ]

    genders_map = dict(name_gender_pairs)

    for name, gender in name_gender_pairs:
        for name_component in name.split():
            genders_map[name_component] = gender

    pd.DataFrame(data={
        "name": genders_map.keys(),
        "gender": genders_map.values(),
    })\
        .set_index("name")\
        .to_csv(genders_map_file)

genders_map

{'Stephan Bonnar': 'man',
 'Ronda Rousey': 'woman',
 'Clint Hester': 'man',
 'Ketlen Vieira': 'woman',
 'Tatiana Suarez': 'woman',
 'Andreas Michailidis': 'man',
 'Jumabieke Tuerxun': 'man',
 'Neil Magny': 'man',
 'Derek Brunson': 'man',
 'Rafael Fiziev': 'man',
 'Pedro Munhoz': 'man',
 'Matt Serra': 'man',
 'Jose Aldo': 'man',
 'Dave Menne': 'man',
 'Renato Moicano': 'man',
 'Ken Shamrock': 'man',
 'Sheldon Westcott': 'man',
 'Cain Velasquez': 'man',
 'Ian McCall': 'man',
 'Zhang Weili': 'woman',
 'Alex Pereira': 'man',
 'Mayra Bueno Silva': 'woman',
 'Michael Bisping': 'man',
 'Rose Namajunas': 'woman',
 'Deiveson Figueiredo': 'man',
 'Ryan Jimmo': 'man',
 'Tito Ortiz': 'man',
 'Ilia Topuria': 'man',
 'Gilbert Burns': 'man',
 'Sean Strickland': 'man',
 'Islam Makhachev': 'man',
 'Max Holloway': 'man',
 'Arman Tsarukyan': 'man',
 'Chris Leben': 'man',
 'Robert Whittaker': 'man',
 'Israel Adesanya': 'man',
 'Michael Chandler': 'man',
 'Antonio Rogerio Nogueira': 'man',
 'Jalin Turner':

In [31]:
def get_coreference_genders(row: pd.Series):
    doc: Doc = row["doc"]

    chains: ChainHolder = doc._.coref_chains
    
    def loop():
        for chain in chains:
            main_item_mention = chain[chain.most_specific_mention_index]
            main_item = doc[main_item_mention.root_index]
            
            if main_item.text in genders_map:
                for mention in chain:
                    token = doc[mention.root_index]
                    yield (token, genders_map[main_item.text])
    
    return list(loop())

In [32]:
df["coreference_genders"] = df.apply(get_coreference_genders, axis="columns")

In [33]:
def get_people_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    people: list[Span] = row["people"]
    coreference_genders: list[tuple[Token, str]] = row["coreference_genders"]

    people_with_gender = [p for p in people if genders_map.get(p.text) == gender]
    coreferences_with_gender = [p for p, g in coreference_genders if g == gender]
    coreferences_with_gender = [Span(doc, p.i, p.i + 1) for p in coreferences_with_gender]
    
    return set(people_with_gender + coreferences_with_gender)

In [34]:
df["men"] = df.apply(get_people_with_gender, args=("man",), axis="columns")
df["women"] = df.apply(get_people_with_gender, args=("woman",), axis="columns")

In [35]:
def get_paragraph_boundaries(doc: Doc):
    line_breaks = [token for token in doc if token.text == "\n"]
    line_break_indexes = [lb.i for lb in line_breaks]
    paragraph_boundaries = list(zip([0] + line_break_indexes, line_break_indexes + [len(doc)]))
    return paragraph_boundaries

In [36]:
df["paragraph_boundaries"] = df["doc"].apply(get_paragraph_boundaries)

In [37]:
def count_genders_in_paragraphs(row: pd.Series, gender: str):
    people_with_gender: set[Span] = row[gender]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]

    def count_gender_in_paragraph(boundary: tuple[int, int]):
        start, end = boundary
        count = 0

        for person in people_with_gender:
            if person.end >= end:
                continue
            elif person.start < start:
                continue
            
            count += 1

        return count

    return [count_gender_in_paragraph(boundary) for boundary in paragraph_boundaries]

In [38]:
df["men_per_paragraph"] = df.apply(count_genders_in_paragraphs, args=("men",), axis="columns")
df["women_per_paragraph"] = df.apply(count_genders_in_paragraphs, args=("women",), axis="columns")

In [39]:
def assign_gender_to_paragraphs(row: pd.Series):
    men_per_paragraph: list[int] = row["men_per_paragraph"]
    women_per_paragraph: list[int] = row["women_per_paragraph"]

    def choose_gender(m: int, f: int):
        if m == 0 and f == 0:
            return None
        elif m > f:
            return "man"
        elif f > m:
            return "woman"
        else: 
            return "equal"

    return [choose_gender(m, f) for m, f in zip(men_per_paragraph, women_per_paragraph)]

In [40]:
df["paragraph_genders"] = df.apply(assign_gender_to_paragraphs, axis="columns")

In [41]:
def get_paragraphs_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]
    paragraph_genders: list[str | None] = row["paragraph_genders"]

    paragraphs_indexes_with_gender = [i for i, g in enumerate(paragraph_genders) if g == gender]
    paragraphs_boundaries_with_gender = [paragraph_boundaries[i] for i in paragraphs_indexes_with_gender]
    paragraphs_with_gender = [doc[start:end] for start, end in paragraphs_boundaries_with_gender]

    return paragraphs_with_gender

In [42]:
df["man_paragraphs"] = df.apply(get_paragraphs_with_gender, args=("man",), axis="columns")
df["woman_paragraphs"] = df.apply(get_paragraphs_with_gender, args=("woman",), axis="columns")
df["equal_paragraphs"] = df.apply(get_paragraphs_with_gender, args=("equal",), axis="columns")
df["genderless_paragraphs"] = df.apply(get_paragraphs_with_gender, args=(None,), axis="columns")

In [43]:
df

,file,text,doc,people,coreference_genders,men,women,paragraph_boundaries,men_per_paragraph,women_per_paragraph,paragraph_genders,man_paragraphs,woman_paragraphs,equal_paragraphs,genderless_paragraphs
0,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ...","(Buckle, up, ,, kids, ,, because, the, June, s...","[(Mario, Bautista), (Bautista), (Ricky, Simon)...","[(Bautista, man), (Bautista, man), (him, man),...","{(Moreno), (Makhachev), (Bautista), (Nurmagome...","{(she), (Namajunas), (Amanda, Ribas), (Namajun...","[(0, 14), (14, 94), (94, 164), (164, 174), (17...","[0, 0, 0, 0, 0, 2, 8, 3, 3, 0, 0, 0, 0, 0, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 9, 3, 0, ...","[None, None, None, None, None, man, man, man, ...","[(\n, A, couple, fights, before, the, bantamwe...","[(\n, Julianna, Peña, seeks, to, successfully,...",[],"[(Buckle, up, ,, kids, ,, because, the, June, ..."
1,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...,"(There, are, all, kinds, of, different, ways, ...","[(Bo, Nickal), (Gerald, Meerschaert), (Kevin, ...","[(Holland, man), (his, man), (he, man), (Nicka...","{(He), (Deiveson, Figueiredo), (his), (Figueir...","{(Mayra, Bueno, Silva), (her), (Andrade), (Sil...","[(0, 41), (41, 78), (78, 133), (133, 172), (17...","[0, 0, 0, 0, 0, 0, 1, 5, 2, 1, 4, 12, 10, 4, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, ...","[None, None, None, None, None, None, man, man,...","[(\n, While, both, Reinier, de, Ridder, and, B...","[(\n, Jessica, Andrade, and, Jasmine, Jasudavi...",[],"[(There, are, all, kinds, of, different, ways,..."
2,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu...","(On, Saturday, ,, February, 8, ,, the, UFC, re...","[(Alexander, Volkanovski), (Adesanya), (Dan, H...","[(Weili, woman), (her, woman), (Velasquez, man...","{(Sean, Strickland), (his), (Gamrot), (themsel...","{(her), (she), (Raquel, Pennington), (her), (h...","[(0, 30), (30, 67), (67, 159), (159, 220), (22...","[0, 0, 4, 2, 4, 7, 5, 4, 8, 3, 3, 0, 0, 0, 0, ...","[0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, ...","[None, None, man, woman, man, man, man, man, m...","[(\n, Over, the, years, ,, the, stops, in, Syd...","[(\n, As, we, ready, to, see, Zhang, Weili, de...",[],"[(On, Saturday, ,, February, 8, ,, the, UFC, r..."
3,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...,"(Fourteen, individuals, have, held, the, UFC, ...","[(Dave, Menne), (Gil, Castillo), (Dricus, Du, ...","[(Franklin, man), (Franklin, man), (Quarry, ma...","{(Whittaker), (he), (Sean, Strickland), (his),...","{(Silva), (he), (Silva), (he), (Silva), (Silva...","[(0, 67), (67, 118), (118, 166), (166, 181), (...","[4, 0, 0, 3, 0, 4, 7, 2, 7, 3, 0, 3, 1, 7, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 3, 2, ...","[man, None, None, man, None, man, man, man, ma...","[(Fourteen, individuals, have, held, the, UFC,...","[(\n, It, ’s, weird, to, try, to, think, back,...","[(\n, Anderson, Silva, defeats, Chael, Sonnen,...","[(\n, While, it, never, profiled, as, the, gla..."
4,news_10-most-memorable-ufc-moments-canada.txt,As we continue our Canadian-focused march thro...,"(As, we, continue, our, Canadian, -, focused, ...","[(Matt, Serra), (Serra), (Liddell), (Tito, Ort...","[(Serra, man), (Serra, man), (his, man), (He, ...","{(his), (Hominick), (Randy, Couture), (Frankli...","{(Irene, Aldana), (Aldana), (Aldana), (Aldana)}","[(0, 47), (47, 67), (67, 115), (115, 144), (14...","[0, 0, 0, 0, 0, 2, 0, 4, 5, 6, 3, 4, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[None, None, None, None, None, man, None, man,...","[(\n, UFC, 83, marked, the, promotion, ’s, fir...","[(\n, When, I, spoke, with, Nunes, a, couple, ...",[],"[(As, we, continue, our, Canadian, -, focused,..."


In [44]:
def save_separate_paragraphs(row: pd.Series):
    original_file: str = row["file"]
    new_folder = original_file.replace(".txt", "")

    os.makedirs(f"./articles/{new_folder}", exist_ok=True)

    man_paragraphs: list[Span] = row["man_paragraphs"]
    man_text = "\n".join([p.text for p in man_paragraphs])

    with open(f"./articles/{new_folder}/man.txt", "w", encoding="utf8") as file:
        file.write(man_text)

    woman_paragraphs: list[Span] = row["woman_paragraphs"]
    woman_text = "\n".join([p.text for p in woman_paragraphs])

    with open(f"./articles/{new_folder}/woman.txt", "w", encoding="utf8") as file:
        file.write(woman_text)

    equal_paragraphs: list[Span] = row["equal_paragraphs"]
    equal_text = "\n".join([p.text for p in equal_paragraphs])

    with open(f"./articles/{new_folder}/equal.txt", "w", encoding="utf8") as file:
        file.write(equal_text)

    genderless_paragraphs: list[Span] = row["genderless_paragraphs"]
    genderless_text = "\n".join([p.text for p in genderless_paragraphs])

    with open(f"./articles/{new_folder}/genderless.txt", "w", encoding="utf8") as file:
        file.write(genderless_text)

In [45]:
df.apply(save_separate_paragraphs, axis="columns")

0    None
1    None
2    None
3    None
4    None
dtype: object